# Blender Dataset — Visualizations

### Import Libraries

In [ ]:
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '1'
import cv2
import sys
import random
import numpy as np
from pathlib import Path
BASE = Path.cwd()
OUT_FOLDER = BASE / 'output'
import matplotlib.pyplot as plt
%matplotlib inline
import matplotlib.image as mpimg
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.patches import Rectangle
from scipy.ndimage import binary_dilation
from utils.common_utils import distinct_colors
from utils.mapping_utils import load_exr, depth_min3x3, map_points

MIN_OBJ_ID = 7
NUM_VIEWS = 3

### Common Functions

In [ ]:
def is_image(p: Path) -> bool:
    return p.is_file() and p.suffix.lower() == '.png'

def groups_of_3_by_folder(root: Path) -> list[tuple[Path, list[Path]]]:
    """
    Returns a list of tuples, where each tuple contains a folder path and a list of 3 image paths from that folder.
    """
    groups = []
    for folder in sorted([p for p in root.rglob('*') if p.is_dir()]):
        imgs = sorted([p for p in folder.iterdir() if is_image(p)])
        for i in range(0, len(imgs), 3):
            chunk = imgs[i:i+3]
            if len(chunk) == 3:
                groups.append((folder, chunk))
    return groups

groups = groups_of_3_by_folder(OUT_FOLDER)

---
## 1. Triplet Visualization

In [ ]:
def show_triplet(triplet: list[Path]) -> None:
    """
    Displays the 3 images in a triplet side by side with their names as titles.
    """
    fig, axes = plt.subplots(1, 3, figsize = (18, 6))
    for ax, p in zip(axes, triplet):
        img = mpimg.imread(p)
        ax.imshow(img)
        ax.set_title(f'View: {p.name[6]}', fontsize = 24, fontweight = 'bold')
        ax.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
folder, triplet = random.choice(groups)
show_triplet(triplet)

---
## 2. Single-View Visualization

In [ ]:
def find_views_in_folder(folder: Path) -> list[int]:
    """
    Finds available view indices by looking for RGB .png files.
    Assumes RGB files are in the folder and ordered as triplets, as in your current code.
    If your RGB naming is different, adapt this accordingly.
    """
    imgs = sorted([p for p in folder.iterdir() if is_image(p)])
    return list(range(len(imgs)))

def show_single_view_row(folder: Path, view_idx: int | None = None, random_view: bool = True) -> int:
    """
    Plots a single row with 3 columns: RGB | Depth | Object Mask
    for one selected view in the given scene folder.

    Returns the selected view index.
    """
    rgb_imgs = sorted([p for p in folder.iterdir() if is_image(p)])
    if not rgb_imgs:
        raise FileNotFoundError(f'No RGB .png images found in: {folder}')

    if view_idx is None:
        if random_view:
            view_idx = random.randrange(len(rgb_imgs))
        else:
            view_idx = 0

    if not (0 <= view_idx < len(rgb_imgs)):
        raise ValueError(f'view_idx must be in [0, {len(rgb_imgs)-1}], got {view_idx}')

    rgb_path = rgb_imgs[view_idx]
    depth_path = folder / f'depth-{view_idx:04d}.exr'
    mask_path  = folder / f'obj_mask_for_view-{view_idx:04d}.exr'

    fig, axes = plt.subplots(1, 3, figsize = (21, 7))
    fig.subplots_adjust(wspace = 0.05)

    # --- Col 0: RGB ---
    rgb = mpimg.imread(rgb_path)
    axes[0].imshow(rgb)
    axes[0].set_title(f'RGB', fontsize = 24, fontweight = 'bold', pad = 8)
    axes[0].axis('off')

    # --- Col 1: Depth ---
    if depth_path.exists():
        depth = load_exr(depth_path)
        if depth.ndim == 3:
            depth = depth[:, :, 0]
        finite_mask = np.isfinite(depth) & (depth < 1e9)
        display_depth = np.where(finite_mask, depth, np.nan)
        axes[1].imshow(display_depth, cmap = 'plasma_r')
        axes[1].set_title('Depth Map', fontsize = 24, fontweight = 'bold', pad = 8)
    else:
        axes[1].text(0.5, 0.5, 'N/A', ha = 'center', va = 'center', fontsize = 24)
        axes[1].set_title('Depth Map', fontsize = 24, fontweight = 'bold', pad = 8)
        print(f'Warning: Depth map not found: {depth_path}')
    axes[1].axis('off')

    # --- Col 2: Object mask ---
    if mask_path.exists():
        mask = load_exr(mask_path)
        if mask.ndim == 3:
            mask = mask[:, :, 0]
        axes[2].imshow(mask, cmap = 'tab20', interpolation = 'nearest')
        axes[2].set_title('Object Mask', fontsize = 24, fontweight = 'bold', pad = 8)
    else:
        axes[2].text(0.5, 0.5, 'N/A', ha = 'center', va = 'center', fontsize = 24)
        axes[2].set_title('Object Mask', fontsize = 24, fontweight = 'bold', pad = 8)
        print(f'Warning: Object mask not found: {mask_path}')
    axes[2].axis('off')

    plt.show()
    return view_idx

In [ ]:
folder, triplet = random.choice(groups)
show_single_view_row(folder, view_idx = 1)

---
## 3. Scene Evolution Visualization

In [ ]:
def plot_scene_triplet_and_topdown(
    folder,
    *,
    room_area: float = 4500.0,
    margin: float = 15.0,
    figsize = (21, 13),
    show: bool = True,
):
    """
    Plot 2-row figure:
      Row 0: render images for each view (3 columns by default)
      Row 1: top-down map for each view

    Args:
        folder: Path or str to the scene folder containing render*.png, camera*.npz, objs_per_view_*.npz
        room_area: room square area (default 4500)
        margin: axis padding margin
        figsize: matplotlib figure size
        show: if True calls plt.show()

    Returns:
        fig, axes
    """
    folder = Path(folder)

    # --- images ---
    img_paths = [folder / f'render{v}.png' for v in range(NUM_VIEWS)]
    if not all(p.exists() for p in img_paths):
        pngs = sorted(folder.glob('*.png'))
        if len(pngs) >= NUM_VIEWS:
            img_paths = pngs[:NUM_VIEWS]
        else:
            raise FileNotFoundError(f'Not enough PNGs in {folder} (need {NUM_VIEWS}).')

    # --- cameras ---
    cam_world = []
    for v in range(NUM_VIEWS):
        cam_path = folder / f'camera{v}.npz'
        if not cam_path.exists():
            raise FileNotFoundError(f'Missing camera file: {cam_path}')
        cam = np.load(str(cam_path))
        R, t = cam['R'], cam['t'].reshape(3)
        cam_world.append(-R.T @ t)

    # --- object poses ---
    poses_per_view = []
    for v in range(NUM_VIEWS):
        pose_path = folder / f'objs_per_view_{v}.npz'
        if not pose_path.exists():
            raise FileNotFoundError(f'Missing pose file: {pose_path}')
        poses_per_view.append(np.load(pose_path, allow_pickle = True)['poses'].item())

    obj_positions_per_view = []
    for v in range(NUM_VIEWS):
        view_objs = {
            int(k): np.array(d['t'])
            for k, d in poses_per_view[v].items()
            if int(k) >= MIN_OBJ_ID
        }
        obj_positions_per_view.append(view_objs)

    all_obj_ids = sorted(set().union(*[d.keys() for d in obj_positions_per_view]))

    # --- colors ---
    obj_colors = {oid: c for oid, c in zip(all_obj_ids, distinct_colors(len(all_obj_ids)))}
    cam_markers = ['^', 's', 'D'][:NUM_VIEWS]

    # --- room square: centered at origin, fixed area ---
    room_side = float(room_area) ** 0.5
    half_side = room_side / 2.0
    cx, cy = 0.0, 0.0

    # --- axis limits ---
    all_x = [cw[0] for cw in cam_world] + [cx - half_side, cx + half_side]
    all_y = [cw[1] for cw in cam_world] + [cy - half_side, cy + half_side]
    xlim = (min(all_x) - margin, max(all_x) + margin)
    ylim = (min(all_y) - margin, max(all_y) + margin)

    # --- figure layout ---
    plt.rcParams.update({
        'font.family': 'sans-serif',
        'axes.spines.top': False,
        'axes.spines.right': False
    })

    fig, axes = plt.subplots(
        2, NUM_VIEWS,
        figsize = figsize,
        gridspec_kw = {'height_ratios': [1, 1.15], 'hspace': 0.08, 'wspace': 0.06},
    )
    fig.patch.set_facecolor('#ffffff')

    # Row 0: renders
    for v in range(NUM_VIEWS):
        ax = axes[0, v]
        ax.imshow(mpimg.imread(img_paths[v]))
        ax.axis('off')
        ax.set_title(f'View: {v}', fontsize = 24, fontweight = 'bold', pad = 6)
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(3)

    # Row 1: top-down
    for v in range(NUM_VIEWS):
        ax = axes[1, v]
        ax.set_facecolor('#ffffff')

        # Room
        ax.add_patch(Rectangle(
            (cx - half_side, cy - half_side), 2 * half_side, 2 * half_side,
            facecolor = '#e8edf2', edgecolor = '#8899aa',
            linewidth = 1.8, linestyle = '--', zorder = 1, alpha = 0.7
        ))
        ax.plot(cx, cy, '+', color = '#8899aa', ms = 10, mew = 1.5, zorder = 2)

        # Other cameras (dimmed)
        for other in range(NUM_VIEWS):
            if other == v:
                continue
            cw_o = cam_world[other]
            ax.scatter(
                cw_o[0], cw_o[1],
                color = '#bbbbbb', marker = cam_markers[other],
                s = 110, zorder = 4, edgecolors = 'white', linewidths = 0.8, alpha = 0.55
            )

        # Objects visible in this view
        for oid, t in obj_positions_per_view[v].items():
            col = obj_colors[oid]
            ax.scatter(t[0], t[1], color = '#333333', s = 200, zorder = 4, alpha = 0.15)
            ax.scatter(t[0], t[1], color = col, s = 140, edgecolors = 'white',
                       linewidths = 1.0, zorder = 5)
            ax.annotate(
                str(oid), xy = (t[0], t[1]),
                xytext = (0, 9), textcoords = 'offset points',
                fontsize = 8.5, ha = 'center', fontweight = 'bold',
                color = col, zorder = 6
            )

        # Active camera
        cw = cam_world[v]
        ax.scatter(cw[0], cw[1], s = 480, color = '#000000', alpha = 0.18, zorder = 6, linewidths = 0)
        ax.scatter(cw[0], cw[1], marker = cam_markers[v], s = 230, zorder = 7,
                   facecolors = '#000000', edgecolors = 'white', linewidths = 1.4)
        ax.annotate(
            f'cam{v}', xy = (cw[0], cw[1]),
            xytext = (0, 14), textcoords = 'offset points',
            fontsize = 10, fontweight = 'bold', ha = 'center', zorder = 8
        )

        # Arrow toward center
        ax.annotate(
            '', xy = (cx, cy), xytext = (cw[0], cw[1]),
            arrowprops = dict(arrowstyle = '->', lw = 2.0, shrinkA = 8, shrinkB = 8, mutation_scale = 16),
            zorder = 6
        )

        ax.set_xlim(xlim)
        ax.set_ylim(ylim)
        ax.set_aspect('equal')
        ax.set_xlabel('X  (Blender units)', fontsize = 14, color = '#000000')
        ax.set_ylabel('Y  (Blender units)', fontsize = 14, color = '#000000')
        ax.tick_params(labelsize = 8, color = '#aaaaaa', labelcolor = '#000000')
        ax.grid(True, alpha = 0.25, color = '#aabbcc', linewidth = 0.7)
        for spine in ax.spines.values():
            spine.set_edgecolor('#cccccc')

    # Shared object legend
    legend_handles = [
        Line2D([0], [0], marker = 'o', color = 'w',
               markerfacecolor = obj_colors[oid], markeredgecolor = 'white',
               markersize = 14, label = f'Obj. {oid}')
        for oid in all_obj_ids
    ]
    fig.legend(
        handles = legend_handles,
        loc = 'lower center',
        ncol = min(len(legend_handles), 10),
        fontsize = 14,
        framealpha = 0.85,
        edgecolor = '#cccccc',
        bbox_to_anchor = (0.5, 0.0),
        frameon = False,
        handlelength = 2.0,
        handleheight = 1.5,
        borderpad = 1.2,
        labelspacing = 1.2,
    )

    if show:
        plt.show()

    return fig, axes

In [ ]:
folder, triplet = random.choice(groups)
fig, axes = plot_scene_triplet_and_topdown(folder)

---
## 4. Overlapping Region Visualization

In [ ]:
def compute_and_plot_overlaps(
    folder,
    *,
    pairs = None,
    show: bool = True,
):
    """
    Load renders/depths/masks/poses/cameras from `folder`,
    compute overlap regions for view pairs, and visualize them.

    Requires these functions in scope:
      - load_exr(path)
      - depth_min3x3(depth)
      - map_points(...)
      - distinct_colors(n)

    Returns:
      overlap_data: dict[pair_idx] -> dict[obj_id] = (src_overlap_bool, tgt_overlap_bool)
    """
    folder = Path(folder)
    if pairs is None:
        pairs = [(0, 1), (1, 2)]

    # ---------- Load all data for the views ----------
    renders, depths, masks, poses, cameras = [], [], [], [], []

    for v in range(NUM_VIEWS):
        renders.append(plt.imread(str(folder / f'render{v}.png')))

        depths.append(load_exr(folder / f'depth-{v:04d}.exr'))

        obj_mask = load_exr(folder / f'obj_mask_for_view-{v:04d}.exr')
        masks.append(np.rint(obj_mask).astype(np.int32))

        poses.append(
            np.load(folder / f'objs_per_view_{v}.npz', allow_pickle = True)['poses'].item()
        )
        cameras.append(np.load(str(folder / f'camera{v}.npz')))

    H, W = depths[0].shape

    # ---------- Compute overlapping regions for each pair ----------
    overlap_data = {}

    for pair_idx, (va, vb) in enumerate(pairs):
        print(f'\n--- Pair render{va} ↔ render{vb} ---')

        depth_a_occ = depth_min3x3(depths[va])
        depth_b_occ = depth_min3x3(depths[vb])

        obj_ids_a = set(np.unique(masks[va]))
        obj_ids_b = set(np.unique(masks[vb]))
        common_ids = sorted((obj_ids_a & obj_ids_b) - set(range(MIN_OBJ_ID)))

        pair_results = {}

        for obj_id in common_ids:
            obj_id_str = str(obj_id)

            # Skip if no pose entry
            if obj_id_str not in poses[va] or obj_id_str not in poses[vb]:
                continue

            mask_a = (masks[va] == obj_id)
            mask_b = (masks[vb] == obj_id)

            n_pixels_a = int(np.sum(mask_a))
            n_pixels_b = int(np.sum(mask_b))
            if n_pixels_a == 0 or n_pixels_b == 0:
                continue

            # --- Forward: va -> vb ---
            coords_a = np.argwhere(mask_a)  # (N, 2) (v,u)
            valid_fwd, mapped_fwd = map_points(
                obj_pass_idx_str = obj_id_str,
                coords = coords_a,
                depth0 = depths[va],
                depth1_occ = depth_b_occ,
                poses0 = poses[va],
                poses1 = poses[vb],
                camera0 = cameras[va],
                camera1 = cameras[vb],
                check_occlusion = True,
            )

            # keep only points that land on same obj in target
            valid_fwd_idx = np.where(valid_fwd)[0]
            if len(valid_fwd_idx) > 0:
                u_mapped = mapped_fwd[valid_fwd_idx, 0].astype(np.int32)
                v_mapped = mapped_fwd[valid_fwd_idx, 1].astype(np.int32)
                lands_on_obj = mask_b[v_mapped, u_mapped]
                valid_fwd[valid_fwd_idx[~lands_on_obj]] = False

            src_overlap = np.zeros((H, W), dtype = bool)
            src_overlap[coords_a[valid_fwd, 0], coords_a[valid_fwd, 1]] = True

            # --- Backward: vb -> va ---
            coords_b = np.argwhere(mask_b)
            valid_bwd, mapped_bwd = map_points(
                obj_pass_idx_str = obj_id_str,
                coords = coords_b,
                depth0 = depths[vb],
                depth1_occ = depth_a_occ,
                poses0 = poses[vb],
                poses1 = poses[va],
                camera0 = cameras[vb],
                camera1 = cameras[va],
                check_occlusion = True,
            )

            valid_bwd_idx = np.where(valid_bwd)[0]
            if len(valid_bwd_idx) > 0:
                u_mapped = mapped_bwd[valid_bwd_idx, 0].astype(np.int32)
                v_mapped = mapped_bwd[valid_bwd_idx, 1].astype(np.int32)
                lands_on_obj = mask_a[v_mapped, u_mapped]
                valid_bwd[valid_bwd_idx[~lands_on_obj]] = False

            tgt_overlap = np.zeros((H, W), dtype = bool)
            tgt_overlap[coords_b[valid_bwd, 0], coords_b[valid_bwd, 1]] = True

            n_src = int(src_overlap.sum())
            n_tgt = int(tgt_overlap.sum())
            if n_src == 0 and n_tgt == 0:
                continue

            pair_results[obj_id] = (src_overlap, tgt_overlap)

            coverage_fwd = n_src / n_pixels_a
            coverage_bwd = n_tgt / n_pixels_b
            min_cov = min(coverage_fwd, coverage_bwd)

            print(
                f'  obj {obj_id}: fwd {n_src}/{n_pixels_a} ({100*coverage_fwd:.1f}%)  '
                f'bwd {n_tgt}/{n_pixels_b} ({100*coverage_bwd:.1f}%)  '
                f'min_coverage={100*min_cov:.1f}%'
            )

        overlap_data[pair_idx] = pair_results

        # ---------- Visualize ----------
        if show:
            if not pair_results:
                print(f'Pair render{va}-render{vb}: no overlapping objects found.')
                continue

            obj_ids_sorted = sorted(pair_results.keys())
            colors = distinct_colors(len(obj_ids_sorted))
            color_map = {oid: colors[i] for i, oid in enumerate(obj_ids_sorted)}

            silhouette_src = {oid: (masks[va] == oid) for oid in obj_ids_sorted}
            silhouette_tgt = {oid: (masks[vb] == oid) for oid in obj_ids_sorted}

            overlay_src = np.zeros((H, W, 4), dtype = np.float32)
            overlay_tgt = np.zeros((H, W, 4), dtype = np.float32)

            for obj_id in obj_ids_sorted:
                src_overlap, tgt_overlap = pair_results[obj_id]
                r, g, b = color_map[obj_id]

                overlay_src[silhouette_src[obj_id]] = [r, g, b, 0.15]
                overlay_tgt[silhouette_tgt[obj_id]] = [r, g, b, 0.15]

                overlay_src[src_overlap] = [r, g, b, 0.50]
                overlay_tgt[tgt_overlap] = [r, g, b, 0.50]

                for mask_2d, overlay in [(src_overlap, overlay_src), (tgt_overlap, overlay_tgt)]:
                    if mask_2d.any():
                        contour = binary_dilation(mask_2d, iterations = 1) ^ mask_2d
                        overlay[contour] = [r, g, b, 1.0]

            # darken non-object areas
            any_object_src = np.zeros((H, W), dtype = bool)
            any_object_tgt = np.zeros((H, W), dtype = bool)
            for oid in obj_ids_sorted:
                any_object_src |= silhouette_src[oid]
                any_object_tgt |= silhouette_tgt[oid]

            dim_src = np.zeros((H, W, 4), dtype = np.float32)
            dim_tgt = np.zeros((H, W, 4), dtype = np.float32)
            dim_src[~any_object_src] = [0, 0, 0, 0.5]
            dim_tgt[~any_object_tgt] = [0, 0, 0, 0.5]

            fig, ax = plt.subplots(1, 2, figsize = (22, 10))
            fig.subplots_adjust(wspace = -0.05, bottom = 0.08)

            labels = {0: 'View: 0', 1: 'View: 1', 2: 'View: 2'}
            for axi, img, dim, overlay, view_id in [
                (ax[0], renders[va], dim_src, overlay_src, va),
                (ax[1], renders[vb], dim_tgt, overlay_tgt, vb),
            ]:
                axi.imshow(img)
                axi.imshow(dim)
                axi.imshow(overlay)
                axi.set_title(labels.get(view_id, f'View: {view_id}'), fontsize = 24, fontweight = 'bold')
                axi.axis('off')

            legend_handles = [
                Patch(facecolor = (*color_map[oid], 0.5), edgecolor = color_map[oid],
                      linewidth = 1.5, label = f'Obj. {oid}')
                for oid in obj_ids_sorted
            ]
            fig.legend(
                handles = legend_handles,
                loc = 'lower center',
                bbox_to_anchor = (0.5, 0.0),
                ncol = min(len(obj_ids_sorted), 8),
                fontsize = 14,
                handlelength = 2.0,
                handleheight = 1.5,
                borderpad = 1.2,
                labelspacing = 1.2,
                frameon = False
            )

            plt.show()

    return overlap_data

In [ ]:
folder, triplet = random.choice(groups)
overlap_data = compute_and_plot_overlaps(folder)